# Credit Card Fraud Detection using Machine Learning

**IBM SkillsBuild | AICTE | BharatCares**  
**Data Analytics with AI Academic Internship 2026**

**Author:** Venkat Kalyan  
**Date:** September 2026

---

### How to use this notebook in Google Colab
1. Upload `creditcard.csv` using the folder icon on the left
2. Run the cells one by one (or Runtime → Run all)


## 1. Install Required Libraries


In [ ]:
!pip install imbalanced-learn xgboost -q
print("✅ Libraries installed successfully!")


## 2. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_score, recall_score)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
sns.set_palette('husl')
%matplotlib inline

print("✅ All libraries imported successfully!")


## 3. Load the Dataset

Make sure you have uploaded **creditcard.csv** to Colab (left side folder icon → Upload).


In [ ]:
df = pd.read_csv('creditcard.csv')
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


## 4. Dataset Overview


In [ ]:
print("="*60)
print("DATASET INFO")
print("="*60)
df.info()

print("\nMissing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nStatistical Summary:")
df.describe()


## 5. Class Distribution (Target Variable)


In [ ]:
class_counts = df['Class'].value_counts()
print("Class Distribution:")
print(class_counts)
print("\nPercentage:")
print(df['Class'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='Class', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class (0 = Legitimate, 1 = Fraud)')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

axes[1].pie(class_counts, labels=['Legitimate (0)', 'Fraud (1)'],
            autopct='%1.3f%%', colors=['#2ecc71', '#e74c3c'],
            explode=(0, 0.1), shadow=True, startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ HIGHLY IMBALANCED DATASET: Only ~0.172% of transactions are fraudulent.")


## 6. Exploratory Data Analysis - Amount & Time


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[df['Class']==0]['Amount'], bins=50, ax=axes[0], color='#2ecc71', kde=True)
axes[0].set_title('Transaction Amount - Legitimate', fontsize=13, fontweight='bold')
axes[0].set_xlim([0, 2500])

sns.histplot(df[df['Class']==1]['Amount'], bins=50, ax=axes[1], color='#e74c3c', kde=True)
axes[1].set_title('Transaction Amount - Fraud', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Legitimate Transactions - Amount Stats:")
print(df[df['Class']==0]['Amount'].describe())
print("\nFraud Transactions - Amount Stats:")
print(df[df['Class']==1]['Amount'].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[df['Class']==0]['Time'], bins=50, ax=axes[0], color='#2ecc71')
axes[0].set_title('Time Distribution - Legitimate', fontsize=13, fontweight='bold')

sns.histplot(df[df['Class']==1]['Time'], bins=50, ax=axes[1], color='#e74c3c')
axes[1].set_title('Time Distribution - Fraud', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


## 7. Correlation Analysis


In [ ]:
corr = df.corr()
class_corr = corr['Class'].sort_values(ascending=False)

print("Top features correlated with Class (Fraud):")
print(class_corr.head(10))
print("\nLeast correlated features:")
print(class_corr.tail(5))

top_features = class_corr.abs().sort_values(ascending=False).head(12).index
plt.figure(figsize=(12, 10))
sns.heatmap(df[top_features].corr(), annot=True, cmap='RdYlBu_r',
            fmt='.2f', linewidths=0.5, center=0)
plt.title('Correlation Heatmap - Top Features with Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Data Preprocessing

- Scale `Amount` and `Time`
- Stratified train-test split
- Apply SMOTE only on training data


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

scaler = RobustScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

print("Features shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTrain class distribution (%):")
print(y_train.value_counts(normalize=True)*100)


In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_res).value_counts())
print("\n✅ SMOTE applied. Test set remains original distribution.")


## 9. Model Building & Evaluation


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("="*60)
    print(f"MODEL: {model_name}")
    print("="*60)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"\nPrecision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")

    return {
        'Model': model_name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'cm': cm
    }

print("✅ Evaluation function ready")


### 9.1 Logistic Regression


In [ ]:
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_res, y_train_res)
lr_results = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")


### 9.2 Random Forest Classifier


In [ ]:
print("Training Random Forest... (this may take 1-2 minutes)")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12,
                                  random_state=42, n_jobs=-1)
rf_model.fit(X_train_res, y_train_res)
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")


### 9.3 XGBoost Classifier


In [ ]:
print("Training XGBoost...")
xgb_model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train_res, y_train_res)
xgb_results = evaluate_model(xgb_model, X_test, y_test, "XGBoost")


## 10. Model Comparison


In [ ]:
results_df = pd.DataFrame([
    {k: v for k, v in lr_results.items() if k not in ['y_pred','y_prob','cm']},
    {k: v for k, v in rf_results.items() if k not in ['y_pred','y_prob','cm']},
    {k: v for k, v in xgb_results.items() if k not in ['y_pred','y_prob','cm']}
])

print("MODEL PERFORMANCE COMPARISON")
print("="*60)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25

axes[0].bar(x - width, results_df.iloc[0][metrics], width, label='Logistic Regression', color='#3498db')
axes[0].bar(x, results_df.iloc[1][metrics], width, label='Random Forest', color='#2ecc71')
axes[0].bar(x + width, results_df.iloc[2][metrics], width, label='XGBoost', color='#e74c3c')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

for res, color, name in [(lr_results,'#3498db','Logistic Regression'),
                         (rf_results,'#2ecc71','Random Forest'),
                         (xgb_results,'#e74c3c','XGBoost')]:
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, color=color, label=f"{name} (AUC={res['ROC-AUC']:.4f})")

axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, res, title in zip(axes, [lr_results, rf_results, xgb_results],
                          ['Logistic Regression', 'Random Forest', 'XGBoost']):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate','Fraud'], yticklabels=['Legitimate','Fraud'])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()


## 11. Feature Importance (XGBoost)


In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(15), palette='viridis')
plt.title('Top 15 Most Important Features (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 10 Important Features:")
print(feature_importance.head(10).to_string(index=False))


## 12. Key Insights & Business Recommendations

### Key Findings
1. **Severe Class Imbalance**: Only ~0.172% of transactions are fraudulent → Accuracy alone is misleading.
2. **SMOTE + Tree-based models** (especially XGBoost) give the best balance of Precision and Recall.
3. PCA features such as **V14, V10, V12, V17** are highly predictive of fraud.
4. Fraudulent transactions show different amount patterns compared to legitimate ones.

### Business Recommendations
- Deploy the best model (usually XGBoost) in a real-time scoring pipeline.
- Tune the decision threshold based on the cost of False Positives vs False Negatives.
- Continuously retrain as new fraud patterns emerge.
- Combine ML scores with rule-based engines for higher coverage.
- Prioritize high **Recall** so that fewer frauds are missed.

### Conclusion
We successfully built and compared three machine learning models for credit card fraud detection.  
**XGBoost** typically delivers the strongest overall performance on this highly imbalanced dataset.

---
**Thank you!**  
IBM SkillsBuild Data Analytics with AI Internship 2026  
**Venkat Kalyan** | BharatCares × AICTE × IBM
